# Phase 1: environment, baselines, and nominal IPPO
This notebook is a command dashboard. The implementation lives in `robosoccer/`; runs are copied to Drive only after metadata says they are complete. Pymunk evaluation is sim-to-sim transfer, not physical deployment.

> **Repaired benchmark:** rerun this notebook from the top. Do not combine pre-repair checkpoints or evaluation tables with the new results. The inexpensive baseline gate must pass before nominal IPPO training begins, and the final Phase-1 audit must pass before opening Phase 2.

## 1–2. Initialization variables

In [1]:
from pathlib import Path
import json, os, shutil, sys
REPO_URL = "https://github.com/djdhillxn/football"
REPO_DIR = Path("/content/robot-soccer-transfer")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "RobotSoccerTransfer"
print(REPO_DIR, DRIVE_PROJECT)

/content/robot-soccer-transfer /content/drive/MyDrive/RobotSoccerTransfer


## 3. Runtime and GPU check

In [2]:
!python --version
!nvidia-smi || true
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

Python 3.12.13
Tue Jul 21 09:47:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------

## 4. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount(str(DRIVE_MOUNT))
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 5. Clone or fast-forward the repository

In [4]:
if REPO_DIR.exists():
    dirty = !git -C {REPO_DIR} status --porcelain
    if dirty:
        raise RuntimeError("Repository has local changes; commit or remove them before pulling.")
    !git -C {REPO_DIR} fetch origin --quiet
    !git -C {REPO_DIR} pull --ff-only
else:
    if REPO_URL.startswith("REPLACE_"):
        raise RuntimeError("Set REPO_URL before cloning.")
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git rev-parse HEAD

Cloning into '/content/robot-soccer-transfer'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 72 (delta 17), reused 69 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 105.87 KiB | 17.65 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/robot-soccer-transfer
16c94cd4ff55ba014fc5865b32a0f2ab380a0110


In [ ]:
%pwd

In [ ]:
%ls

## 6. Install and verify the package

In [5]:
# Preserve Colab's CUDA-enabled Torch, NumPy, and pandas builds.
!python -m pip install -q gymnasium==1.2.1 pettingzoo==1.26.1 pymunk==7.3.0 PyYAML==6.0.3 tensorboard==2.20.0 imageio==2.37.2 imageio-ffmpeg==0.6.0 pytest==8.4.2 ruff==0.14.5
if int(get_ipython().user_ns.get("_exit_code", 0)) != 0:
    raise RuntimeError("dependency installation failed")
!python -m pip install -e . --no-deps -q
if int(get_ipython().user_ns.get("_exit_code", 0)) != 0:
    raise RuntimeError("editable package installation failed")
from importlib.metadata import version
for package in ["robosoccer-transfer", "torch", "numpy", "pandas", "gymnasium", "pettingzoo", "pymunk"]:
    print(package, version(package))
sys.path.insert(0, str(REPO_DIR))
import robosoccer
print("robosoccer", robosoccer.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.1/951.1 kB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.5 MB/s eta 0:00:0000:010:01
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for robosoccer-transfer (pyproject.toml) ... done
robosoccer-transfer 0.1.0
torch 2.11.0+cu128
numpy 2.0.2
pandas 2.2.2
gymnasium 1.2.1
pettingzoo 1.26.1
pymunk 7.3.0
robosoccer 0.1.0


## 7. Shared artifact helpers

In [6]:
def require_shell_success(label):
    status = int(get_ipython().user_ns.get("_exit_code", 0))
    if status != 0:
        raise RuntimeError(f"{label} failed with shell status {status}")

def latest_run(name):
    pointer = REPO_DIR / "runs" / f"latest_{name}.txt"
    if not pointer.is_file():
        raise FileNotFoundError(f"Missing latest-run pointer: {pointer}")
    return Path(pointer.read_text().strip())

def display_json(path):
    data = json.loads(Path(path).read_text())
    print(json.dumps(data, indent=2)[:5000])
    return data

from robosoccer.config import load_config
from robosoccer.evaluation import phase1_readiness_audit

def run_phase1_audit(baseline_run, ippo_run=None):
    baseline_summary = json.loads((Path(baseline_run) / "eval" / "baseline_summary.json").read_text())
    abstract_summary = None
    transfer_summary = None
    if ippo_run is not None:
        abstract_path = Path(ippo_run) / "eval" / "abstract_standard" / "summary.json"
        transfer_path = Path(ippo_run) / "eval" / "pymunk_transfer" / "summary.json"
        abstract_summary = json.loads(abstract_path.read_text()) if abstract_path.is_file() else None
        transfer_summary = json.loads(transfer_path.read_text()) if transfer_path.is_file() else None
    audit = phase1_readiness_audit(
        load_config(REPO_DIR / "configs" / "base.yaml"),
        baseline_summary,
        abstract_summary,
        transfer_summary,
    )
    output_run = Path(ippo_run) if ippo_run is not None else Path(baseline_run)
    audit_path = output_run / "eval" / "phase1_readiness.json"
    audit_path.write_text(json.dumps(audit, indent=2) + "\n")
    print(json.dumps(audit, indent=2))
    return audit

def copy_completed_run(run_dir):
    run_dir = Path(run_dir)
    metadata = json.loads((run_dir / "run_metadata.json").read_text())
    if metadata.get("status") != "complete":
        raise RuntimeError(f"Refusing to sync incomplete run: {run_dir}")
    destination = DRIVE_PROJECT / "runs" / run_dir.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(run_dir, destination, dirs_exist_ok=True)
    return destination

def restore_run(name):
    source = DRIVE_PROJECT / "runs" / name
    destination = REPO_DIR / "runs" / name
    if not source.is_dir():
        raise FileNotFoundError(source)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    metadata = json.loads((destination / "run_metadata.json").read_text())
    pointer = REPO_DIR / "runs" / f"latest_{metadata['experiment_name']}.txt"
    pointer.parent.mkdir(parents=True, exist_ok=True)
    pointer.write_text(str(destination.resolve()) + "\n")
    return destination

def sync_reports_and_runs():
    shutil.copytree(REPO_DIR / "reports", DRIVE_PROJECT / "reports", dirs_exist_ok=True)
    for pointer in (REPO_DIR / "runs").glob("latest_*.txt"):
        shutil.copy2(pointer, DRIVE_PROJECT / pointer.name)

## 8. Run the full test suite

In [7]:
!python -m pytest -q
require_shell_success("pytest")

........................................                                 [100%]
=============================== warnings summary ===============================
tests/test_core.py::test_smoke_training_creates_required_artifacts
tests/test_core.py::test_metadata_is_valid_json_after_smoke
tests/test_core.py::test_run_pointer_created
  /usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
40 passed, 3 warnings in 14.30s


## 9. Run the CPU smoke configuration

In [8]:
!python -m scripts.train --config configs/smoke_test.yaml
require_shell_success("smoke training")
display_json(latest_run("smoke_test") / "run_metadata.json")

INFO: Training MAPPO with 4 environments for 1,024 steps.
MAPPO:   0% 0/4 [00:00<?, ?it/s]INFO: Update 1/4 | success 0.50 | return 2.92 | FPS 122
MAPPO:  25% 1/4 [00:02<00:06,  2.09s/it, kl=0.000, reward=2.92, success=0.50]INFO: Update 2/4 | success 0.14 | return -2.85 | FPS 165
MAPPO:  50% 2/4 [00:03<00:03,  1.89s/it, kl=0.000, reward=-2.85, success=0.14]INFO: Updated failure-directed sampling over 3 profiles.
INFO: Update 3/4 | success 0.00 | return -5.18 | FPS 166
MAPPO:  75% 3/4 [00:04<00:01,  1.38s/it, kl=0.000, reward=-5.18, success=0.00]INFO: Update 4/4 | success 0.14 | return -2.86 | FPS 180
MAPPO: 100% 4/4 [00:06<00:00,  1.41s/it, kl=0.000, reward=-2.86, success=0.14]INFO: Updated failure-directed sampling over 3 profiles.
MAPPO: 100% 4/4 [00:06<00:00,  1.52s/it, kl=0.000, reward=-2.86, success=0.14]
  warnings.warn(

  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

  parse = parser.parseString(pattern)

  parser.resetCache()

  ParserElemen

{'algorithm': 'mappo',
 'command': '/content/robot-soccer-transfer/scripts/train.py --config configs/smoke_test.yaml',
 'cuda_available': True,
 'evaluation_seeds': {'abstract_test': 130000,
  'curriculum': 20000,
  'repair_calibration_abstract': 30000,
  'repair_calibration_transfer': 40000,
  'transfer_test': 140000,
  'validation': 10000,
  'video': 50000},
 'experiment_name': 'smoke_test',
 'failure_exception': None,
 'git_commit': '16c94cd4ff55ba014fc5865b32a0f2ab380a0110',
 'gpu_name': 'NVIDIA L4',
 'output_artifact_paths': {'best_actor': '/content/robot-soccer-transfer/runs/20260721_095003_smoke_test_mappo_seed0/models/best_actor.ts',
  'best_checkpoint': '/content/robot-soccer-transfer/runs/20260721_095003_smoke_test_mappo_seed0/models/best_checkpoint.pt',
  'environment_steps': 1024,
  'final_actor': '/content/robot-soccer-transfer/runs/20260721_095003_smoke_test_mappo_seed0/models/final_actor.ts',
  'final_checkpoint': '/content/robot-soccer-transfer/runs/20260721_095003_smok

## 10. Run the PettingZoo API validation only

In [9]:
!python -m pytest -q tests/test_core.py -k parallel_api
require_shell_success("PettingZoo parallel API tests")

..                                                                       [100%]
2 passed, 38 deselected in 2.93s


## 11. Record a random-policy environment video

In [10]:
!python -m scripts.record_video --config configs/base.yaml --baseline random --simulator abstract --episodes 1
require_shell_success("random-policy video")

INFO: Recorded /content/robot-soccer-transfer/runs/manual_baseline_videos/videos/random_abstract_seed50000_out_of_bounds.mp4


## 12–13. Evaluate scripted baselines and enforce the pre-training gate
This paired-seed, 100-episode evaluation must show a non-saturated, policy-sensitive Pymunk task and useful role coordination before spending GPU time on IPPO.

In [11]:
!python -m scripts.evaluate_baselines --config configs/base.yaml --episodes 100
require_shell_success("baseline evaluation")
baseline_run = latest_run("base")
display_json(baseline_run / "eval" / "baseline_summary.json")
baseline_audit = run_phase1_audit(baseline_run)
if not baseline_audit["baseline_ready"]:
    raise RuntimeError("Baseline gate failed. Do not begin nominal IPPO training.")

Baselines: 100% 599/600 [01:04<00:00,  8.31it/s]WARNING: /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

  parse = parser.parseString(pattern)

  parser.resetCache()

  ParserElement.enablePackrat()

INFO: Completed baseline evaluation: /content/robot-soccer-transfer/runs/20260721_095025_baselines_heuristic_seed0
Baselines: 100% 600/600 [01:05<00:00,  9.11it/s]
{
  "double_chase__abstract": {
    "collision_rate": 1.0,
    "cvar_10_return": -9.760611508254025,
    "episode_count": 100,
    "invalid_action_rate": 0.17024642824607328,
    "mean_action_switches": 12.32,
    "mean_defender_mode_success_rate": 0.15,
    "mean_return": -3.6238620326596664,
    "mean_return_95_ci": [
      -4.629982031867425,
      -2.472015446163805
    ],
    "mean_time_to_score": 6.026666666666668,
    "median_time_to_score": 6.0,


## 14. Train nominal IPPO
Run this only after the baseline gate above passes. This creates a fresh post-repair checkpoint; the older Phase-1 actor is retained only as historical evidence.

In [12]:
!python -m scripts.train --config configs/ippo_nominal.yaml
require_shell_success("nominal IPPO training")

2026-07-21 09:51:36.632768: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 09:51:36.705434: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-07-21 09:51:38,520:jax._src.path:41: etils.epath found. Using etils.epath for file I/O.
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

  parse = parser.parseString(pattern)

  parser.resetCache()

  ParserElement.enablePackrat()

INFO: Training IPPO with 32 environments for 2,000,000 steps.
IPPO:   0% 0/245 [00:00<?, ?it/s]

## 15. Evaluate IPPO in the abstract simulator

In [13]:
ippo_run = latest_run("ippo_nominal")
!python -m scripts.evaluate --run-dir {ippo_run} --checkpoint best --simulator abstract --suite standard
require_shell_success("IPPO abstract evaluation")

Standard: 100% 300/300 [00:23<00:00, 12.68it/s]
INFO: Evaluation complete | abstract standard | episodes 300 | success 0.417


## 16. Evaluate the same frozen IPPO actor in Pymunk

In [14]:
ippo_run = latest_run("ippo_nominal")
!python -m scripts.evaluate --run-dir {ippo_run} --checkpoint best --simulator pymunk --suite transfer
require_shell_success("IPPO Pymunk transfer evaluation")
phase1_audit = run_phase1_audit(baseline_run, ippo_run)
if phase1_audit["phase2_ready"]:
    print("PHASE 2 GO: every Phase-1 safeguard passed.")
else:
    print("PHASE 2 NO-GO: inspect phase1_readiness.json before continuing.")

Transfer: 100% 100/100 [00:17<00:00,  5.65it/s]
INFO: Evaluation complete | pymunk transfer | episodes 100 | success 0.340
{
  "baseline_ready": true,
  "phase2_ready": true,
  "checks": {
    "bounded_fraction_metrics": {
      "passed": true,
      "observed": true,
      "criterion": "every reported fraction is in [0, 1]"
    },
    "baseline_evaluation_complete": {
      "passed": true,
      "observed": {
        "random__abstract": 100,
        "random__pymunk": 100,
        "double_chase__abstract": 100,
        "double_chase__pymunk": 100,
        "role_based__abstract": 100,
        "role_based__pymunk": 100
      },
      "criterion": "each scripted method/simulator cell has at least 100 episodes"
    },
    "pymunk_random_not_saturated": {
      "passed": true,
      "observed": 0.49,
      "criterion": "random Pymunk success <= 0.5"
    },
    "pymunk_policy_sensitive": {
      "passed": true,
      "observed": 0.19000000000000006,
      "criterion": "Pymunk scripted-method

## 17. Record IPPO videos in both simulators

In [15]:
ippo_run = latest_run("ippo_nominal")
!python -m scripts.record_video --run-dir {ippo_run} --checkpoint best --simulator abstract --episodes 3
require_shell_success("IPPO abstract videos")
!python -m scripts.record_video --run-dir {ippo_run} --checkpoint best --simulator pymunk --episodes 3
require_shell_success("IPPO Pymunk videos")

INFO: Recorded /content/robot-soccer-transfer/runs/20260721_095134_ippo_nominal_ippo_seed0/videos/ippo_nominal_abstract_seed50000_success.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_095134_ippo_nominal_ippo_seed0/videos/ippo_nominal_abstract_seed50004_out_of_bounds.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_095134_ippo_nominal_ippo_seed0/videos/ippo_nominal_abstract_seed50001_success.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_095134_ippo_nominal_ippo_seed0/videos/ippo_nominal_pymunk_seed50003_success.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_095134_ippo_nominal_ippo_seed0/videos/ippo_nominal_pymunk_seed50000_out_of_bounds.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_095134_ippo_nominal_ippo_seed0/videos/ippo_nominal_pymunk_seed50001_out_of_bounds.mp4


## 18. Sync completed artifacts to Drive

In [16]:
for name in ["smoke_test", "ippo_nominal"]:
    print(copy_completed_run(latest_run(name)))
print(copy_completed_run(baseline_run))
sync_reports_and_runs()

/content/drive/MyDrive/RobotSoccerTransfer/runs/20260721_095003_smoke_test_mappo_seed0
/content/drive/MyDrive/RobotSoccerTransfer/runs/20260721_095134_ippo_nominal_ippo_seed0
/content/drive/MyDrive/RobotSoccerTransfer/runs/20260721_095025_baselines_heuristic_seed0


## 19. Finished
Verify the Drive copies and `eval/phase1_readiness.json`, then disconnect and delete the Colab runtime. Start Phase 2 only when `phase2_ready` is `true`; full runs are expensive, so do not leave an idle accelerator attached.